# Task 3 — Reconstruct Invoices from Line Items

Since `invoice_number` can't be trusted as a grouping key (Task 1: only ~29% populated; Task 2: at least one number is reused across different customers), define a rule to group line items into real transactions using store + register + customer + time proximity instead, then verify it against actual data rather than trusting it by construction.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw_pos_export.csv")

# parse each row's timestamp format individually (Task 2 finding) --
# letting pandas infer one format for the whole column fails on ~108/131 rows
df["ts_parsed"] = pd.to_datetime(df["transaction_ts"], format="mixed", errors="coerce")

# sort so rows belonging to the same real transaction end up adjacent
df = df.sort_values(["store_id", "register_id", "ts_parsed"]).reset_index(drop=True)

# check: shape unchanged, and only 1 unparsed timestamp (the genuinely blank one)
print("shape:", df.shape)
print("unparsed timestamps:", df["ts_parsed"].isna().sum())

## Reconstruction rule

A new transaction starts whenever the store, register, or customer changes, or more than `GAP_MINUTES` has passed since the previous line item for that same store + register + customer.

In [ ]:
GAP_MINUTES = 2  # max time between line items to count as the same transaction

# time since the previous line item for the same store + register + customer.
# dropna=False keeps walk-ins (blank customer_ref) as their own valid group --
# without it, pandas silently excludes NaN-key rows from grouping, which
# made every walk-in line look like the start of a new transaction
df["prev_ts"] = df.groupby(["store_id", "register_id", "customer_ref"], dropna=False)["ts_parsed"].shift(1)
df["gap_minutes"] = (df["ts_parsed"] - df["prev_ts"]).dt.total_seconds() / 60

# a row starts a new transaction if there's no valid previous row, or too
# much time passed since the last one
df["new_transaction"] = df["prev_ts"].isna() | (df["gap_minutes"] > GAP_MINUTES)

# running count of "new transaction" flags gives every line item a
# transaction id -- NOT invoice_number, since Task 1/2 showed it's missing
# most of the time and can even repeat
df["reconstructed_txn_id"] = df["new_transaction"].cumsum()

# a transaction is flagged for manual review if any of its lines had no
# parseable timestamp -- those rows can't be trusted to sort into the
# right position, so their assigned id may be wrong
df["needs_manual_review"] = df["ts_parsed"].isna()

# check: how many transactions came out of 131 line items
print("distinct reconstructed transactions:", df["reconstructed_txn_id"].nunique())
print("rows flagged needs_manual_review:", df["needs_manual_review"].sum())

## Known edge case: row 63's missing timestamp

Row 63 (a Canvas Tote Bag line, `invoice_number = INV-1009`) has a blank `transaction_ts`. `NaT` values sort to the *end* of their entire `(store_id, register_id)` partition -- not just later than their true neighbors, but past every other customer's rows too. This placed row 63 immediately after an unrelated customer's transaction in the sorted data, and since `reconstructed_txn_id` is assigned positionally (`cumsum()` over the full sorted order), row 63 inherited that unrelated transaction's id instead of its own.

`invoice_number` survives as corroborating evidence even though the timestamp didn't: row 63 shares `INV-1009` with rows 62 and 64, confirming where it actually belongs.

In [ ]:
before_id = df.loc[df["row_id"] == 63, "reconstructed_txn_id"].iloc[0]
correct_id = df.loc[df["row_id"] == 62, "reconstructed_txn_id"].iloc[0]
df.loc[df["row_id"] == 63, "reconstructed_txn_id"] = correct_id

# check: confirm the correction applied, and that row 63 now shares an id
# with its true siblings (rows 62 and 64)
after_id = df.loc[df["row_id"] == 63, "reconstructed_txn_id"].iloc[0]
print(f"row_id 63 reconstructed_txn_id: {before_id} -> {after_id}")
print(df[df["row_id"].isin([62, 63, 64])][["row_id", "customer_ref", "invoice_number", "reconstructed_txn_id"]])

## Build the transaction-level table

In [ ]:
transactions = df.groupby("reconstructed_txn_id").agg(
    store_id=("store_id", "first"),
    register_id=("register_id", "first"),
    customer_ref=("customer_ref", "first"),
    start_ts=("ts_parsed", "min"),
    end_ts=("ts_parsed", "max"),
    n_lines=("row_id", "count"),
    invoice_numbers_seen=("invoice_number", lambda s: sorted(set(x for x in s if isinstance(x, str) and x != ""))),
).reset_index()

review_flag = df.groupby("reconstructed_txn_id")["needs_manual_review"].any()
transactions["needs_manual_review"] = transactions["reconstructed_txn_id"].map(review_flag)

# check: one row per transaction, well below the original 131 line items
print("transactions table shape:", transactions.shape)
transactions.head()

## Final verification

In [ ]:
# invoice numbers that still map to more than 1 transaction --
# should only be INV-1000, a deliberately reused number, not a bug
has_invoice = df[df["invoice_number"].notna() & (df["invoice_number"] != "")]
invoice_check = has_invoice.groupby("invoice_number")["reconstructed_txn_id"].nunique().sort_values(ascending=False)
print("Invoices mapping to >1 transaction (expect only INV-1000):")
print(invoice_check[invoice_check > 1])

# transactions still flagged for manual review after the known correction
print("\nRemaining flagged transactions:")
print(transactions[transactions["needs_manual_review"]])

# row-count sanity check -- these two numbers must match
print(f"\nTotal line items: {len(df)}")
print(f"Sum of n_lines across transactions: {transactions['n_lines'].sum()}")

# open item: also check for transactions containing MORE THAN ONE invoice
# number -- the opposite failure mode (over-merging), not caught by the
# needs_manual_review flag since it only checks for missing timestamps,
# not coarse (date-only) ones. See PROJECT_DOCUMENTATION.md "Open Items".
multi_invoice = transactions[transactions["invoice_numbers_seen"].apply(len) > 1]
print("\nTransactions containing more than one invoice number (check before Task 4):")
print(multi_invoice)

## Known edge cases

| Case | Cause | Resolution |
|---|---|---|
| `INV-1000` maps to 2 transactions | Deliberately planted register-counter-reset scenario | Correct behavior -- rule kept them separate rather than trusting the colliding number |
| Row 63 / `INV-1009` mislabeled | Missing timestamp sorts to end of partition, corrupting positional id assignment | Caught by `needs_manual_review`, corrected using `invoice_number` |
| Possible `ST01` merge of `INV-1007` / `INV-1013` | Date-only timestamps collapse to midnight, losing time-of-day resolution | Open item -- check `multi_invoice` output above; not caught by the missing-timestamp flag |

## Persist to `output/practice.db`

In [ ]:
import sqlite3

conn = sqlite3.connect("../output/practice.db")

# SQLite can't store Python lists directly -- convert to a comma-joined
# string so the column is a plain text type instead. Use a copy so the
# in-notebook `transactions` dataframe keeps the real list type.
transactions_to_write = transactions.copy()
transactions_to_write["invoice_numbers_seen"] = transactions_to_write["invoice_numbers_seen"].apply(
    lambda lst: ", ".join(lst) if lst else ""
)

df.to_sql("staging_line_items", conn, if_exists="replace", index=False)
transactions_to_write.to_sql("staging_transactions", conn, if_exists="replace", index=False)

conn.close()
print("Written to output/practice.db")